# MedCLIP-SAMv2 + MuscleMap Boxes — Evaluation vs Ground Truth

Computes per-muscle metrics for MedSAM segmentations prompted with MuscleMap WB
bounding boxes, for both water and fat-fraction sequences.

| Parameter | Water | Fat Fraction |
|---|---|---|
| Seg dir | `segs_water/` | `segmentations_fat_frac/` |
| Suffix | `_mcsam2boxes.npz` | `_mcsam2boxes.npz` |
| Result dir | `results_water/` | `results_fat_frac/` |
| CSV suffix | `mcsam2boxes_water` | `mcsam2boxes_fatfrac` |

NPZ keys (string): `Gracilis_R`, `Gracilis_L`, `Sartorius_R`, `Sartorius_L`

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1

EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
BASE_DIR = os.path.join(EVAL_DIR, 'medclipsamv2plusboxes')
GT_BASE  = os.path.join(EVAL_DIR, 'myosegmenTUM')

SEG_DIR_WATER = os.path.join(BASE_DIR, 'segs_water')
SEG_DIR_FF    = os.path.join(BASE_DIR, 'segmentations_fat_frac')

# (muscle_name, gt_label_int, npz_key)
MUSCLES = [
    ('R_gracilis',  5, 'Gracilis_R'),
    ('L_gracilis',  1, 'Gracilis_L'),
    ('R_sartorius', 8, 'Sartorius_R'),
    ('L_sartorius', 4, 'Sartorius_L'),
]

print('GT_BASE  :', os.path.abspath(GT_BASE))
print('Water    :', SEG_DIR_WATER, '— exists:', os.path.isdir(SEG_DIR_WATER))
print('FF       :', SEG_DIR_FF,    '— exists:', os.path.isdir(SEG_DIR_FF))

GT_BASE  : C:\Projects\dissector\eval_notebooks\myosegmenTUM
Water    : C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\segs_water — exists: True
FF       : C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\segmentations_fat_frac — exists: True


In [3]:
def parse_subject_stack(filename, modality_tag):
    """
    Extract (subject, stack_num) from filenames like:
      HV001_1_WATER_stack1_mcsam2boxes.npz
      P004_1_FATFRACTION_stack2_mcsam2boxes.npz
    """
    pattern = rf'^(.+)_{modality_tag}_stack(\d+)_mcsam2boxes\.npz$'
    m = re.match(pattern, os.path.basename(filename))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def evaluate_muscle(muscle_name, gt_label_idx, npz_key,
                    seg_files, modality_tag, result_dir, csv_suffix):
    results = []
    for seg_file in seg_files:
        subject, stack_num = parse_subject_stack(seg_file, modality_tag)
        if subject is None:
            print(f'  could not parse: {os.path.basename(seg_file)}, skipping')
            continue

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        data = np.load(seg_file)
        if npz_key not in data.files:
            print(f'  key "{npz_key}" not in {os.path.basename(seg_file)}, skipping')
            print(f'  available keys: {data.files}')
            continue
        pred_arr = data[npz_key].astype(np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {os.path.basename(seg_file)}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'stack':                                stack_num,
            'pred_file':                            os.path.basename(seg_file),
            'gt_path':                              gt_path,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


print('Functions ready.')

Functions ready.


## Water

In [4]:
RESULT_DIR_WATER = os.path.join(BASE_DIR, 'results_water')

seg_files_water = sorted(glob.glob(os.path.join(SEG_DIR_WATER, '*_mcsam2boxes.npz')))
print(f'Found {len(seg_files_water)} water files')
for f in seg_files_water[:5]:
    print(' ', os.path.basename(f))

Found 46 water files
  HV001_1_WATER_stack1_mcsam2boxes.npz
  HV001_1_WATER_stack2_mcsam2boxes.npz
  HV001_2_WATER_stack1_mcsam2boxes.npz
  HV001_2_WATER_stack2_mcsam2boxes.npz
  HV001_3_WATER_stack1_mcsam2boxes.npz


In [5]:
dfs_water = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, key="{npz_key}") ──')
    dfs_water[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        seg_files_water, 'WATER',
        RESULT_DIR_WATER, 'mcsam2boxes_water',
    )
print('\nDone.')


── R_gracilis (gt=5, key="Gracilis_R") ──
  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_water\df_R_gracilis_mcsam2boxes_water.csv

── L_gracilis (gt=1, key="Gracilis_L") ──
  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_water\df_L_gracilis_mcsam2boxes_water.csv

── R_sartorius (gt=8, key="Sartorius_R") ──
  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_water\df_R_sartorius_mcsam2boxes_water.csv

── L_sartorius (gt=4, key="Sartorius_L") ──
  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_water\df_L_sartorius_mcsam2boxes_water.csv

Done.


In [6]:
for name, df in dfs_water.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))


── R_gracilis ──


,subject,stack,R_gracilis_dice,R_gracilis_hausdorff,R_gracilis_jaccard,R_gracilis_boundary_iou_3d
0,HV001_1,1,0.801632,13.601471,0.668937,0.477056
1,HV001_1,2,0.781102,8.000000,0.640827,0.479916
2,HV001_2,1,0.780801,11.401754,0.640421,0.439015
3,HV001_2,2,0.756058,12.000000,0.607792,0.438289
4,HV001_3,1,0.811601,13.453624,0.682936,0.472727
5,HV001_3,2,0.804863,12.000000,0.673448,0.523909
6,HV002_1,1,0.769429,19.416488,0.625262,0.399184
7,HV002_1,2,0.756339,16.000000,0.608155,0.433066
8,HV002_2,1,0.792930,14.142136,0.656904,0.448455
9,HV002_2,2,0.797839,20.000000,0.663670,0.509023



── L_gracilis ──


,subject,stack,L_gracilis_dice,L_gracilis_hausdorff,L_gracilis_jaccard,L_gracilis_boundary_iou_3d
0,HV001_1,1,0.788101,12.083046,0.650303,0.457659
1,HV001_1,2,0.749175,11.704700,0.598944,0.457043
2,HV001_2,1,0.792472,8.062258,0.656277,0.502368
3,HV001_2,2,0.786763,16.000000,0.648482,0.520944
4,HV001_3,1,0.790673,9.848858,0.653812,0.477643
5,HV001_3,2,0.745163,16.000000,0.593833,0.460458
6,HV002_1,1,0.740094,29.274562,0.587420,0.416365
7,HV002_1,2,0.780192,20.000000,0.639602,0.474131
8,HV002_2,1,0.791296,25.000000,0.654665,0.467907
9,HV002_2,2,0.757284,20.000000,0.609378,0.465374



── R_sartorius ──


,subject,stack,R_sartorius_dice,R_sartorius_hausdorff,R_sartorius_jaccard,R_sartorius_boundary_iou_3d
0,HV001_1,1,0.623476,40.049969,0.452935,0.327729
1,HV001_1,2,0.689420,32.015621,0.526042,0.354274
2,HV001_2,1,0.664590,44.045431,0.497668,0.357062
3,HV001_2,2,0.788250,24.020824,0.650506,0.502942
4,HV001_3,1,0.758916,32.015621,0.611495,0.441558
5,HV001_3,2,0.793548,32.124757,0.657754,0.494314
6,HV002_1,1,0.794082,15.099669,0.658488,0.480983
7,HV002_1,2,0.822529,29.068884,0.698556,0.508642
8,HV002_2,1,0.711436,16.763055,0.552115,0.371162
9,HV002_2,2,0.812952,24.515301,0.684851,0.489791



── L_sartorius ──


,subject,stack,L_sartorius_dice,L_sartorius_hausdorff,L_sartorius_jaccard,L_sartorius_boundary_iou_3d
0,HV001_1,1,0.749723,24.041631,0.599645,0.367426
1,HV001_1,2,0.766590,36.069378,0.621521,0.460061
2,HV001_2,1,0.769118,40.012498,0.624852,0.427628
3,HV001_2,2,0.828545,36.069378,0.707279,0.556052
4,HV001_3,1,0.794435,28.000000,0.658973,0.489694
5,HV001_3,2,0.808744,36.110940,0.678901,0.520096
6,HV002_1,1,0.845160,9.899495,0.731842,0.559782
7,HV002_1,2,0.818223,28.142495,0.692367,0.500497
8,HV002_2,1,0.839014,9.000000,0.722674,0.537350
9,HV002_2,2,0.842983,20.784610,0.728583,0.548254


## Fat Fraction

In [7]:
RESULT_DIR_FF = os.path.join(BASE_DIR, 'results_fat_frac')

seg_files_ff = sorted(glob.glob(os.path.join(SEG_DIR_FF, '*_mcsam2boxes.npz')))
print(f'Found {len(seg_files_ff)} fat fraction files')
for f in seg_files_ff[:5]:
    print(' ', os.path.basename(f))

Found 54 fat fraction files
  HV001_1_FATFRACTION_stack1_mcsam2boxes.npz
  HV001_1_FATFRACTION_stack2_mcsam2boxes.npz
  HV001_2_FATFRACTION_stack1_mcsam2boxes.npz
  HV001_2_FATFRACTION_stack2_mcsam2boxes.npz
  HV001_3_FATFRACTION_stack1_mcsam2boxes.npz


In [8]:
dfs_ff = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, key="{npz_key}") ──')
    dfs_ff[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        seg_files_ff, 'FATFRACTION',
        RESULT_DIR_FF, 'mcsam2boxes_fatfrac',
    )
print('\nDone.')


── R_gracilis (gt=5, key="Gracilis_R") ──
  Saved 54 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_fat_frac\df_R_gracilis_mcsam2boxes_fatfrac.csv

── L_gracilis (gt=1, key="Gracilis_L") ──
  Saved 54 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_fat_frac\df_L_gracilis_mcsam2boxes_fatfrac.csv

── R_sartorius (gt=8, key="Sartorius_R") ──
  Saved 54 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_fat_frac\df_R_sartorius_mcsam2boxes_fatfrac.csv

── L_sartorius (gt=4, key="Sartorius_L") ──
  Saved 54 rows -> C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_fat_frac\df_L_sartorius_mcsam2boxes_fatfrac.csv

Done.


In [9]:
for name, df in dfs_ff.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))


── R_gracilis ──


,subject,stack,R_gracilis_dice,R_gracilis_hausdorff,R_gracilis_jaccard,R_gracilis_boundary_iou_3d
0,HV001_1,1,0.704524,12.369317,0.543834,0.502997
1,HV001_1,2,0.748924,16.031220,0.598624,0.516590
2,HV001_2,1,0.725209,14.317821,0.568885,0.479797
3,HV001_2,2,0.692297,20.000000,0.529399,0.475018
4,HV001_3,1,0.663653,12.688578,0.496617,0.462460
5,HV001_3,2,0.720341,16.124515,0.562916,0.488888
6,HV002_1,1,0.655994,23.937418,0.488088,0.405479
7,HV002_1,2,0.665548,24.000000,0.498743,0.510886
8,HV002_2,1,0.710206,19.442222,0.550635,0.467151
9,HV002_2,2,0.749890,24.083189,0.599859,0.532193



── L_gracilis ──


,subject,stack,L_gracilis_dice,L_gracilis_hausdorff,L_gracilis_jaccard,L_gracilis_boundary_iou_3d
0,HV001_1,1,0.686164,19.235384,0.522259,0.442624
1,HV001_1,2,0.706047,13.416408,0.545651,0.463666
2,HV001_2,1,0.717355,12.649111,0.559277,0.505554
3,HV001_2,2,0.721718,20.000000,0.564601,0.484639
4,HV001_3,1,0.652287,18.000000,0.483995,0.431559
5,HV001_3,2,0.702848,17.549929,0.541839,0.460469
6,HV002_1,1,0.625533,32.140317,0.455109,0.389908
7,HV002_1,2,0.663548,24.000000,0.496500,0.435427
8,HV002_2,1,0.625743,23.537205,0.455332,0.388046
9,HV002_2,2,0.677468,24.000000,0.512251,0.441616



── R_sartorius ──


,subject,stack,R_sartorius_dice,R_sartorius_hausdorff,R_sartorius_jaccard,R_sartorius_boundary_iou_3d
0,HV001_1,1,0.676887,40.012498,0.511586,0.375669
1,HV001_1,2,0.685988,20.322401,0.522056,0.444742
2,HV001_2,1,0.660324,44.192760,0.492898,0.376069
3,HV001_2,2,0.759829,20.396078,0.612681,0.515250
4,HV001_3,1,0.742801,36.000000,0.590837,0.438949
5,HV001_3,2,0.771442,20.124612,0.627925,0.544795
6,HV002_1,1,0.759683,13.747727,0.612491,0.469536
7,HV002_1,2,0.750757,25.495098,0.600969,0.498120
8,HV002_2,1,0.710947,17.233688,0.551526,0.453758
9,HV002_2,2,0.752203,20.639767,0.602824,0.495740



── L_sartorius ──


,subject,stack,L_sartorius_dice,L_sartorius_hausdorff,L_sartorius_jaccard,L_sartorius_boundary_iou_3d
0,HV001_1,1,0.751994,20.124612,0.602556,0.464822
1,HV001_1,2,0.733181,44.283180,0.578757,0.475201
2,HV001_2,1,0.743538,40.012498,0.591772,0.475096
3,HV001_2,2,0.783719,36.276714,0.644357,0.546753
4,HV001_3,1,0.781700,28.301943,0.641632,0.502648
5,HV001_3,2,0.775565,32.403703,0.633407,0.513760
6,HV002_1,1,0.816022,11.704700,0.689221,0.565701
7,HV002_1,2,0.783043,28.513155,0.643443,0.519127
8,HV002_2,1,0.789137,10.049876,0.651715,0.515015
9,HV002_2,2,0.759427,24.020824,0.612159,0.496884
